# LangGraph Parallel Workflows

**VidTrace course reconstruction — Agentic AI Class, 24 Aug 2026**

The extracted lecture demonstrates:
1. Non-LLM parallel workflow — Cricket Inning Analysis
2. LLM-style parallel workflow — Essay Evaluation

The cricket example contains `runs=72`, `balls=50`, `fours=8`, `sixes=2`, with independent calculations for strike rate, boundary percentage, and balls per boundary.

In [ ]:
!pip -q install -U langgraph langchain-core

from typing import TypedDict, Annotated
from operator import add
from pprint import pprint
from langgraph.graph import StateGraph, START, END

print("LangGraph setup complete.")

## 1. Non-LLM Parallel Workflow — Cricket Inning Analysis

Conceptual flow extracted from the lecture:

`Initial State → [Strike Rate, Boundary %, Balls / Boundary] → Aggregate`

The three analysis branches do not depend on one another, so they can be started from the same input state.

In [ ]:
class CricketState(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int
    results: Annotated[list[str], add]

def strike_rate(state: CricketState):
    value = state["runs"] / state["balls"] * 100
    return {"results": [f"Strike Rate = {value:.2f}"]}

def boundary_percentage(state: CricketState):
    boundary_runs = state["fours"] * 4 + state["sixes"] * 6
    value = boundary_runs / state["runs"] * 100
    return {"results": [f"Boundary % = {value:.2f}"]}

def balls_per_boundary(state: CricketState):
    boundaries = state["fours"] + state["sixes"]
    value = state["balls"] / boundaries
    return {"results": [f"Balls / Boundary = {value:.2f}"]}

g = StateGraph(CricketState)
g.add_node("strike_rate", strike_rate)
g.add_node("boundary_percentage", boundary_percentage)
g.add_node("balls_per_boundary", balls_per_boundary)

g.add_edge(START, "strike_rate")
g.add_edge(START, "boundary_percentage")
g.add_edge(START, "balls_per_boundary")

g.add_edge("strike_rate", END)
g.add_edge("boundary_percentage", END)
g.add_edge("balls_per_boundary", END)

cricket_graph = g.compile()

cricket_state = {
    "runs": 72,
    "balls": 50,
    "fours": 8,
    "sixes": 2,
    "results": []
}

cricket_result = cricket_graph.invoke(cricket_state)
pprint(cricket_result)

### Why the reducer?

The lecture specifically discusses a reducer when multiple parallel nodes contribute to a common piece of state. Here:

`Annotated[list[str], add]`

means each branch returns a list and the lists are combined.

In [ ]:
print("Aggregated parallel results:")
for item in cricket_result["results"]:
    print(" -", item)

## 2. LLM-Style Parallel Workflow — Essay Evaluation

The extracted slide shows the essay going to three independent evaluators:
- Clarity
- Depth
- Grammar

Then the outputs flow to a final evaluation stage.

The extracted state schema contains feedback and score fields for all three dimensions plus `summary_feedback` and `final_score`.

In [ ]:
class EssayState(TypedDict):
    essay: str
    clarity_feedback: str | None
    clarity_score: int | None
    depth_feedback: str | None
    depth_score: int | None
    grammar_feedback: str | None
    grammar_score: int | None
    summary_feedback: str | None
    final_score: float | None

def evaluate_clarity(state: EssayState):
    return {
        "clarity_feedback": "The ideas are generally easy to follow.",
        "clarity_score": 7
    }

def evaluate_depth(state: EssayState):
    return {
        "depth_feedback": "The essay provides supporting ideas.",
        "depth_score": 6
    }

def evaluate_grammar(state: EssayState):
    return {
        "grammar_feedback": "Grammar is generally clear.",
        "grammar_score": 8
    }

def final_evaluation(state: EssayState):
    scores = [
        state["clarity_score"],
        state["depth_score"],
        state["grammar_score"],
    ]
    return {
        "summary_feedback": "Combined feedback from clarity, depth and grammar.",
        "final_score": round(sum(scores) / len(scores), 2)
    }

e = StateGraph(EssayState)
e.add_node("essay_input", lambda s: s)
e.add_node("clarity", evaluate_clarity)
e.add_node("depth", evaluate_depth)
e.add_node("grammar", evaluate_grammar)
e.add_node("final_evaluation", final_evaluation)

e.add_edge(START, "essay_input")
e.add_edge("essay_input", "clarity")
e.add_edge("essay_input", "depth")
e.add_edge("essay_input", "grammar")

e.add_edge("clarity", "final_evaluation")
e.add_edge("depth", "final_evaluation")
e.add_edge("grammar", "final_evaluation")
e.add_edge("final_evaluation", END)

essay_graph = e.compile()

essay_result = essay_graph.invoke({
    "essay": "India in the Age of AI",
    "clarity_feedback": None,
    "clarity_score": None,
    "depth_feedback": None,
    "depth_score": None,
    "grammar_feedback": None,
    "grammar_score": None,
    "summary_feedback": None,
    "final_score": None
})

pprint(essay_result)

## Key concepts from the lecture

- Parallel nodes receive the same relevant input state.
- Independent branches can execute without waiting for one another.
- A reducer is useful when multiple branches update a shared state field.
- The essay example finishes by combining the three individual evaluation scores.

In [ ]:
assert cricket_result["results"]
assert essay_result["final_score"] == 7.0
print("Both parallel examples completed successfully.")